<a href="https://colab.research.google.com/github/student880/Baymax_Project/blob/main/Copy_of_Baymax_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = "YOUR_KAGGLE_KEY_HERE"
!pip install -q --upgrade kaggle
print("Downloading dataset... (This may take 1-2 minutes)")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
print("Unzipping images...")
!unzip -q chest-xray-pneumonia.zip
print("✅ SUCCESS!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.0/160.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 2.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 5, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 4, in <module>
    from kaggle.api.kaggle_api_extended import KaggleApi
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 55, in <module>
    from kagglesdk import get_access_token_from_env, KaggleClient, KaggleCre

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
train_dir = 'chest_xray/train'
normal_dir = os.path.join(train_dir, 'NORMAL')
pneumonia_dir = os.path.join(train_dir, 'PNEUMONIA')

normal_img_name = random.choice(os.listdir(normal_dir))
pneumonia_img_name = random.choice(os.listdir(pneumonia_dir))

normal_img_path = os.path.join(normal_dir, normal_img_name)
pneumonia_img_path = os.path.join(pneumonia_dir, pneumonia_img_name)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

img1 = mpimg.imread(normal_img_path)
ax[0].imshow(img1, cmap='gray')
ax[0].set_title("NORMAL Lungs")
ax[0].axis('off')

img2 = mpimg.imread(pneumonia_img_path)
ax[1].imshow(img2, cmap='gray')
ax[1].set_title("PNEUMONIA Lungs")
ax[1].axis('off')

plt.show()
print(f"Total Normal Images: {len(os.listdir(normal_dir))}")
print(f"Total Pneumonia Images: {len(os.listdir(pneumonia_dir))}")

FileNotFoundError: [Errno 2] No such file or directory: 'chest_xray/train/NORMAL'

Now the AI wobt be able to look at 5000 images at once cause memory e cover dito na. So we will create an 'Assembly Line' (called Data Generator) that'll feed the images to the AI in small batches (will try to do 32 images at once). will also perform rescalling on the images.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Assembly Line and Rescale
# 80 training, 20 validation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

print("Loading Training Data...")
train_generator = train_datagen.flow_from_directory(
    'chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

print("Loading Validation Data...")
validation_generator = train_datagen.flow_from_directory(
    'chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print("Loading Test Data...")
test_generator = test_datagen.flow_from_directory(
    'chest_xray/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

print("✅ SUCCESS! Bach Geya bhaiya")

now we'll use  a pre-built architecture.
decided to go with ResNet
think of this as hiring an experience doctor who already knows what shapes (of lungs) look like, and we will teach him to specialize in lungs (identifying between vala and pneumonia wala)
The whole process is called Transfer Learning

In [ ]:
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("✅ SUCCESS! ")
model.summary()

now we will force the model to look at our xrays and learn. We are going to run it for 10 epochs. 1 epochs means the model will look at the entire dataset 10times. Why choose 10? cause amra jodi 10er beshi nei taile overfitting howar chances ase.

In [ ]:

print("Starting training...")
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    steps_per_epoch=len(train_generator),
    validation_steps=len(validation_generator)
)
print("✅ SUCCESS!")

In [ ]:
print("Running the Final Exam (Test Set)...")

test_loss, test_acc = model.evaluate(test_generator)

print("\n--------------------------------------")
print(f"Final Test Accuracy: {test_acc * 100:.2f}%")
print("--------------------------------------")

if test_acc > 0.80:
    print("✅ GRADE: A (Excellent)")
elif test_acc > 0.70:
    print("⚠️ GRADE: B (Good, but could be better)")
else:
    print("❌ GRADE: F (Needs retraining)")

In [ ]:
from tensorflow.keras.optimizers import Adam
base_model.trainable = True
fine_tune_at = 150

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
model.compile(optimizer=Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

print(f"✅ SUCCESS!Number of trainable layers: {len(base_model.layers) - fine_tune_at}")
model.summary()

In [ ]:

print("Starting Fine-Tuning...")

history_fine = model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator,
    steps_per_epoch=len(train_generator),
    validation_steps=len(validation_generator)
)

print("✅ SUCCESS!")

model.save('/content/drive/My Drive/Phase1_Project/my_pneumonia_model_finetuned.keras')
print("✅ SUCCESS! Final model saved to Drive.")

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Final Test Accuracy: {test_acc * 100:.2f}%")

now we develop the "Lab Analyst" component of our AI Agent to interpret structured medical data. We utilize a real world Complete Blood Count (CBC) dataset, focusing on key features such as White Blood Cells (WBC), Hemoglobin (HGB), and Platelets (PLT). Since the raw dataset consists only of numerical values, we generate "Ground Truth" diagnostic labels by applying standard medical thresholds—classifying patients as having an Infection (WBC > 11.0), Anemia (HGB < 13.5), or being Healthy. To process this data, we train a Random Forest Classifier, an algorithm well-suited for medical diagnostics due to its ability to aggregate multiple decision trees for high accuracy. The final trained model is saved as blood_analysis_model.pkl and will serve as the logic engine for our agent's lab report interpretation.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

os.environ['KAGGLE_API_TOKEN'] = "KGAT_acb4e37118d7c34397bdb34f3db75fd3"
print("Downloading Real Medical Data...")
!kaggle datasets download -d ahmedelsayedtaha/complete-blood-count-cbc-test --force
!unzip -o -q complete-blood-count-cbc-test.zip
df = pd.read_excel("cbc information.xlsx")
df = df[['WBC', 'HGB', 'PLT']]

def classify_patient(row):
    if row['WBC'] > 11.0:
        return "Infection"
    elif row['HGB'] < 13.5:
        return "Anemia"
    else:
        return "Healthy"
df['Diagnosis'] = df.apply(classify_patient, axis=1)
X = df[['WBC', 'HGB', 'PLT']]
y = df['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training the Blood Analyst AI...")
blood_model = RandomForestClassifier(n_estimators=100)
blood_model.fit(X_train, y_train)

acc = accuracy_score(y_test, blood_model.predict(X_test))
print(f"✅ SUCCESS! Blood Model Accuracy on Real Data: {acc * 100:.2f}%")

joblib.dump(blood_model, '/content/drive/My Drive/Phase1_Project/blood_analysis_model.pkl')
print("✅ Model saved to Drive as 'blood_analysis_model.pkl'")

In [ ]:
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import joblib
import os

from google.colab import drive
drive.mount('/content/drive')

class MedicalAgent:
    def __init__(self):
        print("🤖 Initializing AI Agent...")

        base_path = '/content/drive/My Drive/Phase1_Project/'

        vision_path = os.path.join(base_path, 'my_pneumonia_model_finetuned.keras')
        blood_path = os.path.join(base_path, 'blood_analysis_model.pkl')
        try:
            self.vision_model = load_model(vision_path)
            self.blood_model = joblib.load(blood_path)
            print("✅ SUCCESS: Both AI Models Loaded.")
        except Exception as e:
            print(f"❌ ERROR: Could not load models. Check paths!\n{e}")

    def analyze(self, img_array, wbc, hgb, plt):
        img_array = tf.image.resize(img_array, (224, 224))
        img_array = np.expand_dims(img_array, axis=0) / 255.0

        viz_pred = self.vision_model.predict(img_array, verbose=0)[0][0]
        xray_result = "PNEUMONIA" if viz_pred > 0.5 else "NORMAL"
        confidence = viz_pred if viz_pred > 0.5 else 1 - viz_pred

        df = pd.DataFrame([[wbc, hgb, plt]], columns=['WBC', 'HGB', 'PLT'])
        blood_result = self.blood_model.predict(df)[0]
        report = f"""
        🏥 MEDICAL DIAGNOSTIC REPORT
        ================================
        🩻 X-RAY FINDINGS:
        - Result: {xray_result}
        - Confidence: {confidence*100:.2f}%

        🩸 BLOOD WORK FINDINGS:
        - Diagnosis: {blood_result}

        👨‍⚕️ AGENT RECOMMENDATION:
        """

        if xray_result == "PNEUMONIA" and blood_result == "Infection":
            report += "URGENT: Signs of Pneumonia + Active Infection. Hospitalize immediately."
        elif xray_result == "PNEUMONIA":
            report += "Warning: Pneumonia detected. Verify with CT Scan."
        elif blood_result != "Healthy":
            report += f"Observation: Lungs clear, but blood shows {blood_result}."
        else:
            report += "Patient appears Healthy."

        return report
agent = MedicalAgent()

In [ ]:

# !pip install -q gradio

# # 2. Define the Wrapper Function for the UI
# def run_diagnosis(image, wbc, hgb, plt):
#     if image is None:
#         return "⚠️ Error: Please upload an X-Ray image."
#     return agent.analyze(image, wbc, hgb, plt)
# interface = gr.Interface(
#     fn=run_diagnosis,
#     inputs=[
#         gr.Image(label="Upload Chest X-Ray"),
#         gr.Number(label="White Blood Cells (WBC)", value=6000),
#         gr.Number(label="Hemoglobin (HGB)", value=14.0),
#         gr.Number(label="Platelets (PLT)", value=250000)
#     ],
#     outputs=gr.Textbox(label="AI Agent Report", lines=10),
#     title="🏥 Medical AI Agent",
#     description="Upload an X-Ray and enter blood test values to get a combined diagnosis."
# )

# interface.launch(share=True, debug=True)

In [ ]:
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import joblib
import os
import time

from google.colab import drive
drive.mount('/content/drive')

class MedicalAgent:
    def __init__(self):
        print("🤖 Initializing Baymax...")
        base_path = '/content/drive/My Drive/Phase1_Project/'
        alt_path = '/content/drive/My Drive/Medical_AI_Project/'

        try:
            self.vision_model = load_model(os.path.join(base_path, 'my_pneumonia_model_finetuned.keras'))
            self.blood_model = joblib.load(os.path.join(base_path, 'blood_analysis_model.pkl'))
            print("✅ Loaded from Phase1_Project")
        except:
            print("⚠️ Phase1 path failed, trying Medical_AI_Project...")
            try:
                self.vision_model = load_model(os.path.join(alt_path, 'my_pneumonia_model_finetuned.keras'))
                self.blood_model = joblib.load(os.path.join(alt_path, 'blood_analysis_model.pkl'))
                print("✅ Loaded from Medical_AI_Project")
            except:
                print("❌ CRITICAL ERROR: Models not found. Check your Drive folders!")

    def generate_report(self, img, wbc, hgb, plt_count):
        if img is None:
            return "⚠️ Please upload an X-Ray image to begin analysis."

        # vision analysis
        img_array = tf.image.resize(img, (224, 224))
        img_array = np.expand_dims(img_array.numpy(), axis=0) / 255.0

        # prediction nitesi
        pred = self.vision_model.predict(img_array, verbose=0)[0][0]
        xray_res = "PNEUMONIA" if pred > 0.5 else "NORMAL"
        confidence = pred if pred > 0.5 else 1 - pred

        # blood analysis
        # random forest model er dataframe
        df = pd.DataFrame([[wbc, hgb, plt_count]], columns=['WBC', 'HGB', 'PLT'])
        blood_res = self.blood_model.predict(df)[0]

        # platelet diagnosis
        if plt_count < 150000:
            plt_status = "🔴 Low (Thrombocytopenia)"
            plt_risk = "High risk of bleeding. Coagulation support required."
        elif plt_count > 450000:
            plt_status = "🔴 High (Thrombocytosis)"
            plt_risk = "Increased risk of thrombotic events (clotting)."
        else:
            plt_status = "🟢 Normal"
            plt_risk = "Within normal physiological limits."

        # generating the report
        timestamp = time.strftime("%B %d, %Y - %H:%M")

        #this portion (3lines) is done using AI
        report = f"""
# 🏥 **CLINICAL DIAGNOSTIC REPORT**
**Date:** {timestamp} | **Consultant:** Baymax v2.0

---

## **1. RADIOLOGICAL ASSESSMENT (Chest X-Ray)**
"""
        if xray_res == "PNEUMONIA":
            report += f"""
* **Finding:** 🔴 **POSITIVE FOR PNEUMONIA**
* **AI Confidence:** {confidence*100:.2f}%
* **Analysis:** The computer vision analysis of the thoracic cavity identifies significant opacities consistent with pulmonary consolidation. These visual patterns are indicative of an inflammatory condition affecting the lung parenchyma (alveoli). The heatmap signature correlates strongly with bacterial or viral pneumonia markers observed in the RSNA dataset. **Immediate clinical correlation is advised.**
"""
        else:
            report += f"""
* **Finding:** 🟢 **NORMAL PULMONARY PRESENTATION**
* **AI Confidence:** {confidence*100:.2f}%
* **Analysis:** The thoracic imaging reveals clear lung fields with no obvious evidence of consolidation, pleural effusion, or masses. The cardiac silhouette appears within normal limits, and diaphragmatic contours are sharp and well-defined. No acute radiological abnormalities are detected at this time.
"""

        report += f"""
## **2. HEMATOLOGICAL ASSESSMENT (CBC Panel)**
| Parameter | Value | Reference Range | Status |
| :--- | :--- | :--- | :--- |
| **WBC** | {wbc} /µL | 4,500 - 11,000 | {"🔴 High" if wbc > 11000 else "🔵 Low" if wbc < 4500 else "🟢 Normal"} |
| **Hemoglobin** | {hgb} g/dL | 13.5 - 17.5 | {"🔴 Low (Anemia)" if hgb < 13.5 else "🟢 Normal"} |
| **Platelets** | {plt_count} /µL | 150k - 450k | {plt_status} |

"""
        # diagnosis interpretation
        if blood_res == "Infection":
            report += "**Interpretation:** 🔴 **ACUTE INFECTION DETECTED.** The elevated white blood cell count (Leukocytosis) suggests an active immune response to a pathogen. Combined with radiological findings, this strongly implicates a systemic response."
        elif blood_res == "Anemia":
            report += "**Interpretation:** 🟡 **ANEMIA DETECTED.** Hemoglobin levels are below the standard reference range, indicating reduced oxygen-carrying capacity. This condition may exacerbate respiratory distress."
        else:
            report += "**Interpretation:** 🟢 **HEMATOLOGY NORMAL.** All major blood cell lines appear within standard reference ranges."

        # if platelet count is abnormal
        if "Normal" not in plt_status:
             report += f"\n\n**Platelet Alert:** {plt_risk}"

        report += "\n\n## **3. CLINICAL SYNTHESIS & PLAN**\n"

        # Final Recommendation Logic
        if xray_res == "PNEUMONIA" and blood_res == "Infection":
            report += """
The convergence of radiological evidence (Pneumonia) and hematological markers (Infection) represents a **High-Risk Clinical Profile**.

**Recommended Management Plan:**
1.  **Immediate Hospital Admission:** Patient requires close monitoring of vitals and SpO2.
2.  **Pharmacotherapy:** Initiate broad-spectrum antibiotics immediately, pending blood culture results.
3.  **Respiratory Support:** Administer supplemental oxygen if saturation drops below 92%.
4.  **Diagnostic Expansion:** Consider CT Chest for detailed infiltrate mapping.
"""
        elif xray_res == "PNEUMONIA":
            report += """
Radiological evidence suggests Pneumonia, yet systemic inflammatory markers (WBC) remain normal. This discordance may indicate an early-stage viral pneumonia or atypical presentation.

**Recommended Management Plan:**
1.  **Confirmatory Testing:** Order PCR panel for viral pathogens (Influenza, COVID-19, RSV).
2.  **Outpatient Management:** Prescribe rest, hydration, and antipyretics. Isolate patient.
3.  **Surveillance:** Repeat Chest X-Ray in 48-72 hours to monitor for progression.
"""
        elif blood_res != "Healthy":
            report += f"""
Pulmonary imaging is clear, but the patient exhibits signs of **{blood_res}**. The reported symptoms may be secondary to this underlying condition rather than a lung infection.

**Recommended Management Plan:**
1.  **Specialist Referral:** Consult Hematology or Internal Medicine.
2.  **Additional Labs:** Comprehensive metabolic panel (CMP), Iron studies (if Anemia), or infectious disease panel.
"""
        else:
            report += """
Both imaging and blood work are unremarkable. The patient appears clinically healthy based on the provided diagnostic modalities.

**Recommended Management Plan:**
1.  **Discharge:** No acute medical intervention is required.
2.  **Wellness:** Advise routine annual check-up.
"""
        return report
agent = MedicalAgent()

# the UI using Gradio. used gradio. initially i tried doing an webpage using html, css, and JS, but couldnt proceed with it cause localserver/cloudflare
# were causing problems with tunelling. Then switched to gradio. Why gradio? cause it creates a public link automatically and doesnt require a tunelling password.
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
).set(
    body_background_fill="#0f1117",
    block_background_fill="#161b22",
    block_border_width="0px",
    body_text_color="#c9d1d9",
    block_label_text_color="#8b949e",
    input_background_fill="#0d1117"
)

with gr.Blocks(theme=theme, title="Baymax") as demo:
    gr.Markdown("# ✨ Hello, I am Baymax!\n### Automated Multi-Modal Diagnostic System")

    with gr.Row():
        # input area
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### 1. Patient Data Entry")
            img_input = gr.Image(label="Upload Chest X-Ray", type="numpy", height=300)

            gr.Markdown("### 2. Lab Values (CBC)")
            wbc_input = gr.Number(label="WBC Count", value=6000, info="Normal: 4,500 - 11,000")
            hgb_input = gr.Number(label="Hemoglobin (HGB)", value=14.0, info="Normal: 13.5 - 17.5")
            plt_input = gr.Number(label="Platelets", value=250000, info="Normal: 150k - 450k")

            analyze_btn = gr.Button("✨ Generate Analysis", variant="primary", size="lg")

        # output area
        with gr.Column(scale=2):
            gr.Markdown("### 3. AI Analysis Report")
            output_report = gr.Markdown(label="Clinical Report")

    # connecting the button
    analyze_btn.click(
        fn=agent.generate_report,
        inputs=[img_input, wbc_input, hgb_input, plt_input],
        outputs=output_report
    )

print("🚀 Baymax...")
demo.launch(share=True, debug=True)